# SUFE VOS: SAM 3.1 Object Multiplex Full Predictor

Default checkpoint source: `research21/sam3.1`. The submission path now requires official `build_sam3_multiplex_video_predictor` plus full first-frame mask conditioning. The older low-level tracker/remap path is rejected for submissions and only available as `low_level_debug` in local debugging. Recommended runtime: H100 80GB, then A100 80GB/40GB. T4 is rejected; L4 is suitable only for smoke tests.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib, datetime, subprocess, re, zipfile
DRIVE_ROOT = pathlib.Path(os.environ.get('SUFE_DRIVE_ROOT', '/content/drive/MyDrive'))
def clean_repo_url(value):
    text = str(value or '').strip()
    markdown = re.search(r'\]\((https?://[^)\s]+)\)', text)
    if markdown:
        text = markdown.group(1).strip()
    else:
        plain_url = re.search(r'https?://[^\s)>\]]+', text)
        if plain_url:
            text = plain_url.group(0).strip()
    if text.startswith('<') and text.endswith('>'):
        text = text[1:-1].strip()
    return text.strip("`'\"")
def run_streamed(cmd, label, env=None):
    import os, shlex, subprocess
    child_env = os.environ.copy() if env is None else dict(env)
    child_env.setdefault('PYTHONUNBUFFERED', '1')
    child_env.setdefault('PYTHONIOENCODING', 'utf-8')
    print(f'Running {label}:', ' '.join(str(part) for part in cmd), flush=True)
    shell = globals().get('get_ipython', lambda: None)()
    if shell is not None and hasattr(shell, 'system'):
        command = ' '.join(shlex.quote(str(part)) for part in cmd)
        updates = {str(key): str(value) for key, value in child_env.items() if value is not None}
        previous = {key: os.environ.get(key) for key in updates}
        try:
            os.environ.update(updates)
            shell.system(command)
            returncode = int(shell.user_ns.get('_exit_code', 0) or 0)
        finally:
            for key, value in previous.items():
                if value is None:
                    os.environ.pop(key, None)
                else:
                    os.environ[key] = value
        return subprocess.CompletedProcess(cmd, returncode)
    process = subprocess.Popen(
        cmd,
        stdin=subprocess.DEVNULL,
        stdout=None,
        stderr=None,
        env=child_env,
    )
    return subprocess.CompletedProcess(cmd, process.wait())
def run_git(cmd, label):
    result = run_streamed(cmd, label)
    if result.returncode != 0:
        raise RuntimeError(f'{label} failed with exit code {result.returncode}: {cmd}')
    return result
REPO_URL = clean_repo_url(os.environ.get('SUFE_REPO_URL', 'https://github.com/yyj11266/SUFE_DL_FinalProject_Vision.git'))
PROJECT_ROOT = pathlib.Path(os.environ.get('SUFE_PROJECT_ROOT', '/content/sufe_vos_leaderboard')).expanduser().resolve()
if (PROJECT_ROOT / '.git').exists():
    run_git(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'], 'git pull')
elif not PROJECT_ROOT.exists() and REPO_URL:
    run_git(['git', 'clone', REPO_URL, str(PROJECT_ROOT)], 'git clone')
elif PROJECT_ROOT.exists() and not any(PROJECT_ROOT.iterdir()) and REPO_URL:
    PROJECT_ROOT.rmdir()
    run_git(['git', 'clone', REPO_URL, str(PROJECT_ROOT)], 'git clone')
elif PROJECT_ROOT.exists() and not (PROJECT_ROOT / 'src' / 'data' / 'inspect_sufe.py').exists():
    raise RuntimeError(f'PROJECT_ROOT exists but is not this repo: {PROJECT_ROOT}. Remove it or set SUFE_PROJECT_ROOT.')
DATA_ROOT = pathlib.Path(os.environ.get('SUFE_DATA_ROOT', '/content/sufe_data/video_dataset')).expanduser().resolve()
DATA_ZIP = pathlib.Path(os.environ.get('SUFE_DATA_ZIP', str(DRIVE_ROOT / 'sufe_vos_inputs/video_dataset.zip'))).expanduser().resolve()
OUTPUT_ROOT = pathlib.Path(os.environ.get('SUFE_OUTPUTS_ROOT', '/content/sufe_runs')).expanduser().resolve()
REVIEW_ROOT = pathlib.Path(os.environ.get('SUFE_PUBLISH_ROOT', str(DRIVE_ROOT / 'sufe_vos_review' / 'runs'))).expanduser().resolve()
ARCHIVE_ROOT = pathlib.Path(os.environ.get('SUFE_ARCHIVE_ROOT', str(DRIVE_ROOT / 'sufe_vos_archives'))).expanduser().resolve()
HF_REPO_ID = os.environ.get('SAM31_HF_REPO_ID', 'research21/sam3.1')
CHECKPOINT_FILENAME = os.environ.get('SAM31_CHECKPOINT_FILENAME', 'sam3.1_multiplex.pt')
sample_candidates = [DRIVE_ROOT / 'sufe_vos_inputs/sample_submission.zip', PROJECT_ROOT / 'data/sample_submission.zip']
SAMPLE_SUBMISSION = next((path for path in sample_candidates if path.exists()), sample_candidates[0])
MOSEV2_ROOT = pathlib.Path(os.environ.get('MOSEV2_ROOT', '/content/drive/MyDrive/datasets/MOSEv2'))
checkpoint_candidates = [
    pathlib.Path(os.environ['SAM31_CHECKPOINT']) if os.environ.get('SAM31_CHECKPOINT') else None,
    PROJECT_ROOT / 'checkpoints' / CHECKPOINT_FILENAME,
    OUTPUT_ROOT / 'checkpoints' / CHECKPOINT_FILENAME,
]
checkpoint_candidates.extend(sorted(OUTPUT_ROOT.glob(f'*/checkpoints/{CHECKPOINT_FILENAME}')) if OUTPUT_ROOT.exists() else [])
SAM31_CHECKPOINT = next((path for path in checkpoint_candidates if path and path.exists()), None)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
REVIEW_ROOT.mkdir(parents=True, exist_ok=True)
ARCHIVE_ROOT.mkdir(parents=True, exist_ok=True)
if not DATA_ROOT.exists() or not any(DATA_ROOT.iterdir()):
    if not DATA_ZIP.exists():
        raise RuntimeError(f'DATA_ROOT is missing/empty and DATA_ZIP does not exist: {DATA_ROOT} ; {DATA_ZIP}')
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DATA_ZIP, 'r') as archive:
        archive.extractall(DATA_ROOT)
    print('Extracted dataset to:', DATA_ROOT)
else:
    print('DATA_ROOT already prepared:', DATA_ROOT)
os.chdir(PROJECT_ROOT)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_ROOT:', DATA_ROOT)
print('DATA_ZIP:', DATA_ZIP)
print('MOSEV2_ROOT:', MOSEV2_ROOT)
print('HF_REPO_ID:', HF_REPO_ID)
print('SAM31_CHECKPOINT:', SAM31_CHECKPOINT if SAM31_CHECKPOINT else 'will download from Hugging Face')
print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('REVIEW_ROOT:', REVIEW_ROOT)
print('ARCHIVE_ROOT:', ARCHIVE_ROOT)
def run_checked(cmd, label, exp_dir=None, env=None):
    import pathlib
    result = run_streamed(cmd, label, env=env)
    if exp_dir:
        status_path = pathlib.Path(exp_dir) / 'logs/per_video_status.json'
        sanity_path = pathlib.Path(exp_dir) / 'sanity_check.json'
        for path in (status_path, sanity_path):
            if path.exists():
                print(f'--- {path} ---')
                print(path.read_text(encoding='utf-8')[:12000])
    if result.returncode != 0:
        raise RuntimeError(f'{label} failed with exit code {result.returncode}. See streamed output/status above.')
    return result


In [ ]:
import subprocess, sys, shutil, os
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_ROOT / 'requirements_colab.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'sam3'], check=False)
SAM3_REPO = pathlib.Path('/content/facebookresearch_sam3')
if SAM3_REPO.exists():
    shutil.rmtree(SAM3_REPO)
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/facebookresearch/sam3.git', str(SAM3_REPO)], check=True)
if not (SAM3_REPO / 'sam3/model_builder.py').exists():
    raise RuntimeError(f'Official SAM3 source is incomplete: {SAM3_REPO}; files={list(SAM3_REPO.glob("sam3/*"))[:20]}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation', '-e', str(SAM3_REPO)], check=True)
os.environ['PYTHONPATH'] = str(SAM3_REPO) + os.pathsep + os.environ.get('PYTHONPATH', '')
check_env = os.environ.copy()
subprocess.run([sys.executable, '-c', 'from sam3.model_builder import build_sam3_multiplex_video_predictor; import sam3.model_builder as mb; print("subprocess_sam3_full_predictor_ok", mb.__file__)'], check=True, env=check_env)
print('SAM3 source:', SAM3_REPO)
print('PYTHONPATH head:', os.environ['PYTHONPATH'].split(os.pathsep)[0])


In [ ]:
import subprocess, sys, os, platform
runtime_code = r'''
import torch, platform
from packaging.version import Version
from sam3.model_builder import build_sam3_multiplex_video_predictor
assert Version(torch.__version__.split('+')[0]) >= Version('2.7'), torch.__version__
assert torch.cuda.is_available(), 'CUDA GPU is required'
assert Version(str(torch.version.cuda)) >= Version('12.6'), torch.version.cuda
gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
assert major >= 8 and torch.cuda.is_bf16_supported(), f'BF16 Ampere+ GPU required, got {gpu_name}'
assert 'T4' not in gpu_name.upper(), 'T4 is excluded for this full run'
print({'python': platform.python_version(), 'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu': gpu_name, 'sam3_builder': 'build_sam3_multiplex_video_predictor'})
'''
run_checked([sys.executable, '-c', runtime_code], 'runtime-preflight', env=os.environ.copy())
if SAM31_CHECKPOINT:
    checkpoint = str(SAM31_CHECKPOINT)
else:
    from huggingface_hub import hf_hub_download
    checkpoint = hf_hub_download(repo_id=HF_REPO_ID, filename=CHECKPOINT_FILENAME)
print('checkpoint:', checkpoint, flush=True)


## SAM 3.1 API and State Probe

Run this before full smoke/full inference. It writes compact JSON diagnostics for one short real video so we can verify the official full predictor mask-conditioning state flow.

In [ ]:
import shutil
RUN_SAM31_API_PROBE = os.environ.get('RUN_SAM31_API_PROBE', '1').strip().lower() not in {'0', 'false', 'no'}
PROBE_VIDEO_ID = os.environ.get('SAM31_PROBE_VIDEO_ID', '2b827e3a')
PROBE_MAX_FRAMES = os.environ.get('SAM31_PROBE_MAX_FRAMES', '5')
if RUN_SAM31_API_PROBE:
    PROBE_EXP = 'sam31_api_probe_' + datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    probe_dir = OUTPUT_ROOT / PROBE_EXP
    probe_cmd = [
        sys.executable, str(PROJECT_ROOT / 'scripts/debug_sam31_api.py'),
        '--data-root', str(DATA_ROOT),
        '--output-dir', str(OUTPUT_ROOT),
        '--experiment-id', PROBE_EXP,
        '--video-id', PROBE_VIDEO_ID,
        '--max-frames', PROBE_MAX_FRAMES,
        '--checkpoint', checkpoint,
        '--sam3-repo-dir', str(SAM3_REPO),
        '--multiplex-count', os.environ.get('SAM31_MULTIPLEX_COUNT', '16'),
    ]
    if os.environ.get('SAM31_ALLOW_UNSUPPORTED_RUNTIME', '0').strip().lower() in {'1', 'true', 'yes'}:
        probe_cmd.append('--allow-unsupported-runtime')
    probe_error = None
    try:
        run_checked(probe_cmd, 'sam31-api-probe', exp_dir=probe_dir, env=os.environ.copy())
    except RuntimeError as exc:
        probe_error = exc
    probe_review_dir = REVIEW_ROOT / PROBE_EXP
    probe_review_dir.mkdir(parents=True, exist_ok=True)
    for rel in ['logs/summary.json', 'logs/sam31_api_introspection.json', 'logs/sam31_state_probe.json']:
        src = probe_dir / rel
        if src.exists():
            dst = probe_review_dir / rel
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
    print('SAM3.1 API probe review diagnostics:', probe_review_dir)
    for name in ['summary.json', 'sam31_api_introspection.json', 'sam31_state_probe.json']:
        path = probe_dir / 'logs' / name
        if path.exists():
            print(f'--- {path} ---')
            print(path.read_text(encoding='utf-8')[:12000])
    if probe_error is not None:
        raise probe_error
else:
    print('RUN_SAM31_API_PROBE=0; skipped SAM 3.1 API/state probe.')


In [ ]:
import json, shutil
from IPython.display import Image as DisplayImage, display
smoke_videos = ['0u8fy7u2', '2b827e3a', '2a1jkxdf', 'kpg9gld7', 'lkob5diu', 'pjlde9hu']
SAM31_EMPTY_MASK_POLICY = os.environ.get('SAM31_EMPTY_MASK_POLICY', 'empty').strip().lower()
if SAM31_EMPTY_MASK_POLICY not in {'empty', 'previous'}:
    raise ValueError(f'Unsupported SAM31_EMPTY_MASK_POLICY={SAM31_EMPTY_MASK_POLICY!r}')
SAM31_INDEXED_ABSENCE_POLICY = os.environ.get('SAM31_INDEXED_ABSENCE_POLICY', SAM31_EMPTY_MASK_POLICY).strip().lower()
if SAM31_INDEXED_ABSENCE_POLICY not in {'empty', 'previous'}:
    raise ValueError(f'Unsupported SAM31_INDEXED_ABSENCE_POLICY={SAM31_INDEXED_ABSENCE_POLICY!r}')
SMOKE_EXP = 'sam31_smoke_full_predictor_' + datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
cmd = [
    sys.executable, str(PROJECT_ROOT / 'scripts/run_sam31_vos.py'),
    '--data-root', str(DATA_ROOT),
    '--output-dir', str(OUTPUT_ROOT),
    '--experiment-id', SMOKE_EXP,
    '--checkpoint', checkpoint,
    '--sam3-repo-dir', str(SAM3_REPO),
    '--prompt-mode', 'mask',
    '--sam3-run-mode', 'full_predictor_mask',
    '--original-resolution',
    '--video-ids', ','.join(smoke_videos),
    '--max-frames', '20',
    '--save-native-scores',
    '--save-overlays', 'all',
    '--smoke-quality-gate',
]
if SAM31_EMPTY_MASK_POLICY != 'empty':
    cmd.extend(['--sam3-empty-mask-policy', SAM31_EMPTY_MASK_POLICY])
if SAM31_INDEXED_ABSENCE_POLICY != 'empty':
    cmd.extend(['--sam3-indexed-absence-policy', SAM31_INDEXED_ABSENCE_POLICY])
smoke_dir = OUTPUT_ROOT / SMOKE_EXP
smoke_error = None
try:
    run_checked(cmd, 'smoke-full-predictor', exp_dir=smoke_dir, env=os.environ.copy())
except RuntimeError as exc:
    smoke_error = exc
finally:
    review_dir = REVIEW_ROOT / SMOKE_EXP
    review_dir.mkdir(parents=True, exist_ok=True)
    for rel in ['logs/smoke_quality_gate.json', 'logs/per_video_status.json', 'run_manifest.json', 'sanity_check.json']:
        src = smoke_dir / rel
        if src.exists():
            dst = review_dir / rel
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
    contact_dir = smoke_dir / 'logs/smoke_contact_sheets'
    if contact_dir.exists():
        dst_contact = review_dir / 'logs/smoke_contact_sheets'
        if dst_contact.exists():
            shutil.rmtree(dst_contact)
        shutil.copytree(contact_dir, dst_contact)
    print('Smoke review diagnostics:', review_dir)
smoke_gate_path = smoke_dir / 'logs/smoke_quality_gate.json'
if not smoke_gate_path.exists():
    if smoke_error is not None:
        raise smoke_error
    raise RuntimeError(f'SAM3.1 smoke did not write quality gate diagnostics: {smoke_gate_path}')
smoke_gate = json.loads(smoke_gate_path.read_text())
print(json.dumps({'smoke_exp': SMOKE_EXP, 'passed': smoke_gate.get('passed'), 'errors': smoke_gate.get('errors', [])[:8], 'warnings': smoke_gate.get('warnings', []), 'contact_sheets': smoke_gate.get('contact_sheets', {})}, indent=2))
for path in smoke_gate.get('contact_sheets', {}).values():
    display(DisplayImage(filename=path))
if not smoke_gate['passed']:
    raise RuntimeError(f"SAM3.1 smoke quality gate failed; diagnostics were copied to {review_dir}. Do not run full submission.")


In [ ]:
SPLIT_PATH = OUTPUT_ROOT / 'validation/mosev2_seed2026.json'
if MOSEV2_ROOT.exists():
    split_cmd = [sys.executable, '-m', 'src.eval.mosev2_split', '--root', str(MOSEV2_ROOT), '--output', str(SPLIT_PATH), '--seed', '2026', '--total', '80', '--calibration-size', '40']
    run_checked(split_cmd, 'mosev2-split', env=os.environ.copy())
    for split_name in ('calibration', 'holdout'):
        exp = f'sam31_mosev2_{split_name}_seed2026'
        split_run = [sys.executable, str(PROJECT_ROOT / 'scripts/run_sam31_vos.py'), '--data-root', str(MOSEV2_ROOT), '--output-dir', str(OUTPUT_ROOT), '--experiment-id', exp, '--checkpoint', checkpoint, '--sam3-repo-dir', str(SAM3_REPO), '--prompt-mode', 'mask', '--sam3-run-mode', 'full_predictor_mask', '--original-resolution', '--video-ids-file', str(SPLIT_PATH), '--split', split_name, '--save-native-scores', '--save-overlays', 'sample', '--skip-existing']
        split_dir = OUTPUT_ROOT / exp
        run_checked(split_run, f'mosev2-{split_name}', exp_dir=split_dir, env=os.environ.copy())
else:
    print('MOSEv2 not found; place it at', MOSEV2_ROOT, 'before threshold calibration. SUFE baseline can still run.')


In [ ]:
import json
smoke_gate = json.loads((smoke_dir / 'logs/smoke_quality_gate.json').read_text())
assert smoke_gate['passed'], smoke_gate
if not SAMPLE_SUBMISSION.exists():
    raise RuntimeError(f'SAMPLE_SUBMISSION is required for strict validation before submission: {SAMPLE_SUBMISSION}')
FULL_EXP = 'sam31_native_full_predictor_' + datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
full_cmd = [
    sys.executable, str(PROJECT_ROOT / 'scripts/run_sam31_vos.py'),
    '--data-root', str(DATA_ROOT),
    '--sample-submission', str(SAMPLE_SUBMISSION),
    '--output-dir', str(OUTPUT_ROOT),
    '--experiment-id', FULL_EXP,
    '--checkpoint', checkpoint,
    '--sam3-repo-dir', str(SAM3_REPO),
    '--prompt-mode', 'mask',
    '--sam3-run-mode', 'full_predictor_mask',
    '--original-resolution',
    '--save-native-scores',
    '--save-overlays', 'sample',
    '--skip-existing',
    '--quality-gate',
    '--quality-gate-baseline-exp', os.environ.get('SAM31_QUALITY_GATE_BASELINE_EXP', str(OUTPUT_ROOT / 'data_prep_20260608_065509')),
    '--make-submission',
]
full_dir = OUTPUT_ROOT / FULL_EXP
run_checked(full_cmd, 'full-predictor', exp_dir=full_dir, env=os.environ.copy())


publish_dir = REVIEW_ROOT / FULL_EXP
publish_cmd = [
    sys.executable, str(PROJECT_ROOT / 'scripts/publish_run_for_codex.py'),
    '--exp-dir', str(full_dir),
    '--publish-dir', str(publish_dir),
    '--data-root', str(DATA_ROOT),
    '--archive-dir', str(ARCHIVE_ROOT),
    '--replace',
]
if os.environ.get('SUFE_MAKE_FULL_ARCHIVE', '0').strip().lower() in {'1', 'true', 'yes'}:
    publish_cmd.append('--make-full-archive')
run_checked(publish_cmd, 'publish-codex-review', exp_dir=publish_dir, env=os.environ.copy())
print('Codex review bundle:', publish_dir)


In [ ]:
import json, zipfile
exp_dir = OUTPUT_ROOT / FULL_EXP
status = json.loads((exp_dir / 'logs/per_video_status.json').read_text())
sanity = json.loads((exp_dir / 'sanity_check.json').read_text())
manifest = json.loads((exp_dir / 'run_manifest.json').read_text())
assert manifest['sam3_run_mode'] == 'full_predictor_mask', manifest
assert manifest['mask_conditioning_path'] == 'official_full_predictor_private_tracker_add_new_objects', manifest
assert status['summary']['status'] == 'done', status['summary']
assert status['summary']['num_videos'] == 25, status['summary']
assert status['summary']['num_frames'] == 2015, status['summary']
assert status['summary']['all_first_frames_exact'] is True
assert sanity['passed'] and sanity['num_actual_masks'] == 2015, sanity
with zipfile.ZipFile(exp_dir / 'submission.zip') as archive:
    assert len([name for name in archive.namelist() if name.lower().endswith('.png')]) == 2015
print({'experiment': FULL_EXP, 'submission': str(exp_dir / 'submission.zip'), 'sanity': sanity, 'manifest_mode': manifest['sam3_run_mode']})
